# Lesson 1: Router Engine

# 说明

说有相关资料都在了，本课程所需的包在《Building and Evaluating Advanced RAG Applications》课程已经安装过了

### 代码功能总结
这个代码演示了 **LlamaIndex 的 Router Engine（路由查询引擎）** 功能。其核心思想是：
- 针对同一份文档（metagpt.pdf），创建两种不同类型的索引：**摘要索引 (Summary Index)** 和 **向量索引 (Vector Index)**。
- 创建两个对应的查询引擎，分别用于回答概括性问题和具体细节问题。
- 使用 `RouterQueryEngine` 作为“智能路由器”，它会根据用户提出的问题内容，自动判断并选择最合适的查询引擎来获取答案，从而实现更智能、更准确的问答。

### 主要步骤
1. **环境设置**：加载OpenAI API密钥，并配置异步支持。
2. **加载与分割数据**：读取PDF文档，并使用句子分割器将其分割成大小为1024字符的文本块（nodes）。
3. **定义模型**：全局配置使用的语言模型（LLM）和嵌入模型（Embedding Model）。
4. **创建索引**：
    - `SummaryIndex`: 将所有文本块连接起来，适合生成全文摘要。
    - `VectorStoreIndex`: 为每个文本块生成向量嵌入，适合进行语义相似度搜索。
5. **创建查询引擎工具**：将上述两个索引转换为查询引擎，并用 `QueryEngineTool` 包装，同时提供描述，告诉路由器每个工具的用途。
6. **定义路由引擎**：创建 `RouterQueryEngine`，它内部使用一个 `LLMSingleSelector`（基于LLM的选择器）来分析用户的问题，并从多个工具中选择一个最合适的来执行。
7. **测试与验证**：通过提出不同类型的问题（如要求摘要、询问具体信息）来测试路由引擎是否能正确地选择不同的底层引擎。

Welcome to Lesson 1.

To access the `requirements.txt` file, the data/pdf file required for this lesson and the `helper` and `utils` modules, please go to the `File` menu and select`Open...`.

I hope you enjoy this course!

欢迎来到第一课。

要获取本课程所需的 requirements.txt 文件、数据/PDF 文件以及 helper 和 utils 模块，请前往 文件 菜单并选择 打开...。

希望您享受这门课程！

**不需要做已经有了**


## Setup

In [15]:
from helper import get_openai_api_key, get_dashscope_api_key
import os

In [16]:
import nest_asyncio

# 应用异步支持（在Jupyter环境中必需）
nest_asyncio.apply()

## Load Data

In [17]:
from llama_index.core import SimpleDirectoryReader


# 步骤1：加载文档
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()
print(f"成功加载 {len(documents)} 个文档")

成功加载 29 个文档


## Define LLM and Embedding model
## 定义LLM与嵌入模型

In [18]:
from llama_index.core.node_parser import SentenceSplitter

# 步骤2：分割文档
# 将文档分割成大小为1024个字符的块，便于后续处理
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)
print(f"文档已分割成 {len(nodes)} 个文本块")


文档已分割成 34 个文本块


In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 配置全局设置：指定使用的语言模型和嵌入模型
# 这里用了DashScope的大模型替代OpenAI模型，
# LlamaIndex支持多种LLM接口，DaskScope兼容OpenAI API，可以使用OpenAILike类调用
# LlamaIndex也有专门的DashScope支持包，具体见 https://developers.llamaindex.ai/python/examples/llm/dashscope/
# llm = OpenAI(
#     api_key=get_openai_api_key(),
#     api_base="https://models.inference.ai.azure.com/",
#     model="gpt-4o-mini",
#     temperature=0.1,
#     context_window=128000,
#     is_chat_model=True,
#     is_function_calling_model=False,
# )

llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)

Settings.llm = llm
# 需要openai key所以改成开源模型
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
Settings.embed_model = HuggingFaceEmbedding(
        model_name=model_real_path,
        # 默认情况下，LlamaIndex 会尝试自动下载和加载模型
        device="cpu",  # 如果您没有GPU，可以使用"cpu"
        # 连不了外网下载模型到本地的记得这个标志要设置为True，不然虽然本地有了还会掉huggingface获取包信息检验
        local_files_only=True,
    )

2026-05-05 15:01:59,887 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


## Define Summary Index and Vector Index over the Same Data
## 在相同数据上定义摘要索引与向量索引

In [ ]:
from llama_index.core import SummaryIndex, VectorStoreIndex

# 可以把索引看作是我们数据的一组元数据
# 步骤3：创建两种索引
# - SummaryIndex：用于生成整个文档的摘要
# - VectorStoreIndex：用于语义搜索，查找与问题最相关的文本块
summary_index = SummaryIndex(nodes)
vector_index = VectorStoreIndex(nodes)
print("已创建摘要索引和向量索引")

已创建摘要索引和向量索引


## Define Query Engines and Set Metadata
## 定义查询引擎并设置元数据

In [26]:
# 步骤4：创建查询引擎
# - summary_query_engine：基于摘要索引，适合回答概括性问题
# - vector_query_engine：基于向量索引，适合回答具体细节问题
# 调用 summary_index 的 as_query_engine 方法，将其转换为一个可交互的查询对象
summary_query_engine = summary_index.as_query_engine(
    # 设置响应模式为“树状总结”。
    # 逻辑是：先对底层各个节点分别总结，再将这些总结汇总成更高层的总结，
    # 直到最终生成一个能够覆盖所有相关信息的完整答案。
    response_mode="tree_summarize",
    
    # use_async=True: 开启异步执行模式。
    # 这允许程序同时向 LLM 发起多个总结请求，而不是一个个排队等待。
    # 在处理长文档或大量节点时，这能显著缩短生成答案的总耗时。
    use_async=True,
)
vector_query_engine = vector_index.as_query_engine()
print("已创建两种查询引擎")

已创建两种查询引擎


In [22]:
from llama_index.core.tools import QueryEngineTool

# 步骤5：封装查询引擎为工具
# 为每个查询引擎添加描述，帮助路由器理解其用途
summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to MetaGPT"
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the MetaGPT paper."
    ),
)
print("已将查询引擎封装为工具")

已将查询引擎封装为工具


## Define Router Query Engine
## 定义路由查询引擎

In [27]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# 步骤6：创建路由查询引擎
# RouterQueryEngine会根据问题内容，自动选择最合适的查询引擎
# LLMSingleSelector使用LLM来决定选择哪个工具
query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True  # 开启详细模式，可以看到路由决策过程
)
print("\n路由查询引擎已准备就绪！")


路由查询引擎已准备就绪！


In [28]:
# 步骤7：测试路由功能
print("\n=== 测试1：摘要类问题 文档的总结是什么？ ===")
response = query_engine.query("What is the summary of the document?") # 文档的总结是什么？
print(str(response))


=== 测试1：摘要类问题 文档的总结是什么？ ===


2026-05-05 15:12:06,301 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 15:12:06,303 - INFO - Selecting query engine 0: The question 'What is the summary of the document?' is asking for a summarization, which aligns with choice (1) that mentions being useful for summarization questions..


Selecting query engine 0: The question 'What is the summary of the document?' is asking for a summarization, which aligns with choice (1) that mentions being useful for summarization questions..


2026-05-05 15:12:28,539 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


The document introduces MetaGPT, a meta-programming framework designed to enhance the problem-solving capabilities of multi-agent systems based on Large Language Models (LLMs). It incorporates human-like Standardized Operating Procedures (SOPs) into LLM-based multi-agent collaborations, enabling more efficient and coherent workflows. MetaGPT assigns specific roles to agents, such as Product Manager, Architect, and Engineer, and uses structured communication and an assembly line paradigm to break down complex tasks into manageable subtasks. The framework also includes an executable feedback mechanism to improve code generation during runtime. Experiments show that MetaGPT outperforms existing methods on benchmarks like HumanEval and MBPP, and it demonstrates robustness and efficiency in handling complex software development tasks.


In [29]:
# 查看引用了多少个文档片段
print(len(response.source_nodes))

34


In [30]:
print("\n=== 测试2：具体信息问题 代理如何与其他代理共享信息？===")
response = query_engine.query(
    "How do agents share information with other agents?"  # 代理如何与其他代理共享信息？
)
print(str(response))


=== 测试2：具体信息问题 代理如何与其他代理共享信息？===


2026-05-05 15:12:59,646 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-05 15:12:59,649 - INFO - Selecting query engine 1: The question 'How do agents share information with other agents?' requires specific details about the mechanisms or methods described in the MetaGPT paper, which would be best addressed by retrieving specific context from the paper..


Selecting query engine 1: The question 'How do agents share information with other agents?' requires specific details about the mechanisms or methods described in the MetaGPT paper, which would be best addressed by retrieving specific context from the paper..


2026-05-05 15:13:04,866 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


Agents share information with other agents through a shared message pool. They publish their structured messages in this pool and can also access messages from other entities transparently. This allows any agent to directly retrieve the required information from the shared pool, eliminating the need to inquire about other agents and await their responses. Additionally, there is a subscription mechanism where agents can select information to follow based on their role profiles, ensuring they only receive task-related information and avoid distractions from irrelevant details.


## Let's put everything together
## 整合所有代码到一个方法里

In [31]:
import importlib
import utils
importlib.reload(utils)
from utils import get_router_query_engine

query_engine = get_router_query_engine("metagpt.pdf")

2026-05-05 15:14:47,633 - INFO - Load pretrained SentenceTransformer: /Users/a1-6/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a


In [14]:

print("\n=== 测试3：关于消融研究的问题 请告诉我关于消融研究的结果？===")
response = query_engine.query("Tell me about the ablation study results?")  # 请告诉我关于消融研究的结果？
print(str(response))


=== 测试3：关于消融研究的问题 请告诉我关于消融研究的结果？===


2026-05-05 14:54:11,646 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/chat/completions "HTTP/1.1 200 OK"
2026-05-05 14:54:11,650 - INFO - Selecting query engine 1: The question asks for specific results from an ablation study, which would require retrieving specific context from the MetaGPT paper..


Selecting query engine 1: The question asks for specific results from an ablation study, which would require retrieving specific context from the MetaGPT paper..


2026-05-05 14:54:15,407 - INFO - HTTP Request: POST https://models.inference.ai.azure.com/chat/completions "HTTP/1.1 200 OK"


The provided information does not include specific details about ablation study results. It primarily focuses on executability comparisons, performance metrics of MetaGPT with different LLMs, the impact of instruction levels on performance, and the performance of GPT variants in the HumanEval benchmark. If you need insights on a particular aspect of the study, please specify!
